In [16]:
import pandas as pd
import numpy as np

In [17]:
echonest = pd.read_csv('../data/raw/fma_metadata/echonest.csv', header=[0, 1, 2], index_col=0)
print(echonest.shape)
print(echonest.head())

(13129, 249)
               echonest                                                    \
         audio_features                                                     
           acousticness danceability    energy instrumentalness  liveness   
track_id                                                                    
2              0.416675     0.675894  0.634476         0.010628  0.177647   
3              0.374408     0.528643  0.817461         0.001851  0.105880   
5              0.043567     0.745566  0.701470         0.000697  0.373143   
10             0.951670     0.658179  0.924525         0.965427  0.115474   
134            0.452217     0.513238  0.560410         0.019443  0.096567   

                                                                        ...  \
                                           metadata                     ...   
         speechiness    tempo   valence  album_date         album_name  ...   
track_id                                                

In [18]:
# pulling just valence and energy 
valence = echonest['echonest']['audio_features']['valence']
energy  = echonest['echonest']['audio_features']['energy']


In [19]:
# computing emotion scores
emotion_df = pd.DataFrame({'track_id': echonest.index})
emotion_df['valence'] = valence.values
emotion_df['energy']  = energy.values

emotion_df['emotion_joy_excitement'] = np.sqrt(emotion_df['valence']**2 + emotion_df['energy']**2) / np.sqrt(2)
emotion_df['emotion_peaceful_content'] = np.sqrt(emotion_df['valence']**2 + (1 - emotion_df['energy'])**2) / np.sqrt(2)
emotion_df['emotion_anger_tension'] = np.sqrt((1 - emotion_df['valence'])**2 + emotion_df['energy']**2) / np.sqrt(2)
emotion_df['emotion_sadness'] = np.sqrt((1 - emotion_df['valence'])**2 + (1 - emotion_df['energy'])**2) / np.sqrt(2)


In [20]:
# softmax 
exp_sum = (np.exp(emotion_df['emotion_joy_excitement'])
         + np.exp(emotion_df['emotion_peaceful_content'])
         + np.exp(emotion_df['emotion_anger_tension'])
         + np.exp(emotion_df['emotion_sadness']))

emotion_df['emotion_joy_excitement_softmax'] = np.exp(emotion_df['emotion_joy_excitement']) / exp_sum
emotion_df['emotion_peaceful_content_softmax'] = np.exp(emotion_df['emotion_peaceful_content']) / exp_sum
emotion_df['emotion_anger_tension_softmax'] = np.exp(emotion_df['emotion_anger_tension']) / exp_sum
emotion_df['emotion_sadness_softmax'] = np.exp(emotion_df['emotion_sadness']) / exp_sum


In [21]:
# merging with cleaned FMA dataset
fma_clean = pd.read_csv('../data/cleaned/fma_cleaned_dataset.csv')

# echonest index IS the track_id, so reset it into a column
emotion_df2 = emotion_df.copy()
emotion_df2['track_id'] = echonest.index.astype(int)

fma_labeled = fma_clean.merge(emotion_df2, on='track_id', how='inner')

print(f"Tracks before merge: {len(fma_clean)}")
print(f"Tracks after merge:  {len(fma_labeled)}")
print(fma_labeled.head())

Tracks before merge: 8000
Tracks after merge:  1294
   track_id               title genre_top  \
0         2                Food   Hip-Hop   
1         5          This World   Hip-Hop   
2        10             Freeway       Pop   
3       140  Queen Of The Wires      Folk   
4       141                Ohio      Folk   

                               mp3_path   valence    energy  \
0  ../data/raw/fma_small/000/000002.mp3  0.576661  0.634476   
1  ../data/raw/fma_small/000/000005.mp3  0.621661  0.701470   
2  ../data/raw/fma_small/000/000010.mp3  0.963590  0.924525   
3  ../data/raw/fma_small/000/000140.mp3  0.609991  0.265685   
4  ../data/raw/fma_small/000/000141.mp3  0.163950  0.075632   

   emotion_joy_excitement  emotion_peaceful_content  emotion_anger_tension  \
0                0.606258                  0.482776               0.539340   
1                0.662768                  0.487639               0.563560   
2                0.944260                  0.683448             

In [22]:
fma_labeled.to_csv('../data/cleaned/fma_emotion_labels.csv', index=False)
print("Saved to data/cleaned/fma_emotion_labels.csv")

Saved to data/cleaned/fma_emotion_labels.csv
